# Figure 1d — Bit_1 spot detection

This publication notebook is a cleaned duplicate of the original single-channel
spot-detection working notebook.

In the manuscript, the `A20` imaging channel is named **Bit_1**. This notebook
detects Bit_1 puncta independently of the full-barcode codebook-matching method.

Scope:

- `reg000`: PBS spleen
- `reg001`: SM-102 LNP-treated spleen
- Bit_1 candidate detection, cell assignment, and cell-level CSV calls

The liver region, luciferase/transfection analysis, saved QC images, and unrelated
exploratory outputs have been removed. This notebook is intentionally distributed
unexecuted.


## 1. Environment and packaged inputs

Install the dependency set listed in `Figure_1d_1e_requirements.txt`.

The notebook reads the same two stitched/registered spleen TIFF stacks, marker
lists, tissue masks, and frozen cell-centroid tables as the full-barcode detector:

`../Data/Figure_1d_1e_Spleen_LNP/Raw_Spot_Detection_Input/`

Cell segmentation was performed using the previously published workflow and is a
frozen upstream input here. Running this notebook writes CSV files only to
`../Data/Figure_1d_1e_Spleen_LNP/Generated_Output/Bit_1/`.


In [ ]:
from IPython.display import display
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple
import json
import logging
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile
from scipy import ndimage as ndi
from scipy.spatial import cKDTree

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-5s | %(message)s',
    datefmt='%H:%M:%S',
    stream=sys.stdout,
    force=True,
)
log = logging.getLogger('spot-yining')

def tic():
    return time.perf_counter()

def toc(t0, label=''):
    dt = time.perf_counter() - t0
    log.info(f'{label} done in {dt:.2f} s')
    return dt


def resolve_data_root() -> Path:
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        for candidate in (
            base / "Manuscripts" / "NanoSTAMP" / "Data" / "Figure_1d_1e_Spleen_LNP",
            base / "Data" / "Figure_1d_1e_Spleen_LNP",
        ):
            if candidate.exists():
                return candidate
    raise FileNotFoundError(
        "Could not locate Data/Figure_1d_1e_Spleen_LNP. "
        "Run this notebook inside the supplied NanoSTAMP publication package."
    )


DATA_ROOT = resolve_data_root()
SEG_DIR = DATA_ROOT / "Raw_Spot_Detection_Input"
OUTDIR = DATA_ROOT / "Generated_Output" / "Bit_1"
OUTDIR.mkdir(parents=True, exist_ok=True)

REGIONS = [
    dict(region="reg000", sample="registered1"),
    dict(region="reg001", sample="registered2"),
]

@dataclass
class Params:
    # Spot enhancement and detection.
    log_sigma: float = 1.0
    peak_width: int = 21
    nms_min_distance: int = 9
    threshold_peaks: float = 300 #800

    # Raw/LoG intensity extraction around detected spots.
    max_filter_width: int = 3

    # Hot-pixel masking. For A20-only detection, one saturated channel is enough.
    hotpx_count_threshold: int = 1
    hotpx_sat_value: float = 65500.0

    # Assignment to Mesmer cells from reg*_features.csv centroids.
    # This local version does centroid-based nearest-cell assignment because
    # seg_output contains feature tables, not full per-pixel cell-label masks.
    cell_assignment_radius_px: float = 20.0
    min_spots_per_cell: int = 2

    # Cell-level barcode binarization for downstream clustering.
    # A barcode bit is 1 when the cell has at least this many assigned spots
    # for the corresponding RCA/barcode marker. Start with 1; use 2 for a
    # stricter call in noisier data.
    min_spots_for_bit: int = 1

    # Error-tolerant LNP barcode assignment. A cell can still be called as a
    # library barcode when the observed barcode is within this Hamming distance.
    # For this one-marker A20-only notebook, keep this at 0 so only A20+
    # cells are called as LNP_A.
    barcode_max_hamming_distance: int = 0

    # Optional crop for quick testing. Use None for full image.
    # Example: (slice(0, 4096), slice(0, 4096))
    roi_yx: Optional[Tuple[slice, slice]] = None

P = Params()
log.info(f'DATA_ROOT:    {DATA_ROOT}')
log.info(f'SEG_DIR:      {SEG_DIR}')
log.info(f'OUTDIR:       {OUTDIR}')
log.info(f'params:       {P}')


## 2. Verify stitched/registered inputs

The detector starts from the integrated, registered multichannel spleen stacks.
Only the `A20` channel—called Bit_1 in the manuscript—is used for punctum
detection. Registration summaries and marker lists are supplied beside the TIFFs.


In [ ]:
def read_marker_list(path: Path) -> List[str]:
    return [line.strip() for line in path.read_text().splitlines() if line.strip()]

def region_paths(region: str, sample: str) -> Dict[str, Path]:
    return {
        'image': SEG_DIR / f'{sample}_integrated_registered_overlap_crop.tif',
        'markerlist': SEG_DIR / f'{sample}_integrated_MarkerList.txt',
        'features': SEG_DIR / f'{region}_features.csv',
        'tissue_mask': SEG_DIR / f'{region}_mask.npy',
        'summary': SEG_DIR / f'{sample}_registration_summary.json',
    }

region_info = []
for item in REGIONS:
    paths = region_paths(item['region'], item['sample'])
    missing = [name for name, path in paths.items() if name != 'tissue_mask' and not path.exists()]
    if missing:
        raise FileNotFoundError(f"{item['region']} missing: {missing}")

    markers = read_marker_list(paths['markerlist'])
    with open(paths['summary']) as f:
        summary = json.load(f)
    shape = tuple(summary.get('integrated_shape', (len(markers), None, None)))
    if shape[0] != len(markers):
        log.warning(f"{item['region']}: marker count {len(markers)} != image channels {shape[0]}")

    barcode_markers = [m for m in markers if m.startswith('A')]
    region_info.append({**item, **paths, 'markers': markers, 'shape': shape, 'barcode_markers': barcode_markers})

pd.DataFrame([
    dict(region=r['region'], sample=r['sample'], image_shape=r['shape'], n_markers=len(r['markers']),
         barcode_markers=', '.join(r['barcode_markers']), features=r['features'].name)
    for r in region_info
])


In [ ]:
# Bit_1 corresponds to the A20 image channel in the registered stack.
BARCODE_MARKERS = ["A20"]

if not BARCODE_MARKERS:
    raise ValueError("Bit_1/A20 marker is not configured")

missing_by_region = {
    r["region"]: [marker for marker in BARCODE_MARKERS if marker not in r["markers"]]
    for r in region_info
}
missing_by_region = {
    region: missing for region, missing in missing_by_region.items() if missing
}
if missing_by_region:
    raise ValueError(
        f"Bit_1 detection requires the A20 channel in every marker list: {missing_by_region}"
    )

log.info(f"Bit_1 image channel: {BARCODE_MARKERS[0]}")


In [ ]:
# A20-only barcode library.
# The barcode string order is exactly BARCODE_MARKERS above.
# With BARCODE_MARKERS = ['A20'], barcode '1' means A20-positive LNP_A.
BARCODE_LIBRARY: Dict[str, str] = {
    '1': 'LNP_A',
}

# Which library barcode to use for random-field visual QC panels.
QC_REFERENCE_LNP = 'LNP_A'

# A one-bit barcode should use exact matching only.
P.barcode_max_hamming_distance = 0

def validate_barcode_library():
    expected_len = len(BARCODE_MARKERS)
    names = list(BARCODE_LIBRARY.values())
    if len(names) != len(set(names)):
        raise ValueError('BARCODE_LIBRARY has duplicated LNP names; names should be unique.')
    for barcode, name in BARCODE_LIBRARY.items():
        if len(barcode) != expected_len:
            raise ValueError(
                f'{name} barcode {barcode} has length {len(barcode)}, '
                f'but BARCODE_MARKERS has length {expected_len}.'
            )
        bad = sorted(set(barcode) - {'0', '1'})
        if bad:
            raise ValueError(f'{name} barcode {barcode} contains non-binary symbols: {bad}')

validate_barcode_library()
log.info(f'Barcode bit order: {BARCODE_MARKERS}')
log.info(f'Barcode library entries: {len(BARCODE_LIBRARY)} -> {BARCODE_LIBRARY}')
log.info(f'barcode_max_hamming_distance: {P.barcode_max_hamming_distance}')


In [ ]:
# Random-field visual QC settings.
# Checks if the markers positive/negative from the barcode is consistent with the expected pattern for the reference LNP.
# By default, derive positive/negative marker panels from QC_REFERENCE_LNP.
def marker_sets_for_lnp_name(lnp_name: str) -> Tuple[List[str], List[str]]:
    matches = [barcode for barcode, name in BARCODE_LIBRARY.items() if name == lnp_name]
    if not matches:
        raise ValueError(f'QC_REFERENCE_LNP={lnp_name!r} is not present in BARCODE_LIBRARY')
    barcode = matches[0]
    pos = [marker for marker, bit in zip(BARCODE_MARKERS, barcode) if bit == '1']
    neg = [marker for marker, bit in zip(BARCODE_MARKERS, barcode) if bit == '0']
    return pos, neg

if QC_REFERENCE_LNP is None:
    QC_POSITIVE_MARKERS = list(BARCODE_MARKERS)
    QC_NEGATIVE_MARKERS = []
else:
    QC_POSITIVE_MARKERS, QC_NEGATIVE_MARKERS = marker_sets_for_lnp_name(QC_REFERENCE_LNP)

QC_N_FIELDS_PER_REGION = 6
QC_FIELD_SIZE = 1024
QC_RANDOM_SEED = 7
QC_MIN_TISSUE_FRACTION = 0.10

log.info(f'QC_REFERENCE_LNP: {QC_REFERENCE_LNP}')
log.info(f'QC positive markers: {QC_POSITIVE_MARKERS}')
log.info(f'QC negative markers: {QC_NEGATIVE_MARKERS}')


## 3. Helper functions

The detector applies a Laplacian-of-Gaussian filter to Bit_1, identifies local
maxima, calibrates the detection threshold using PBS and treated spleen fields,
suppresses nearby duplicate maxima, masks saturated pixels, and assigns retained
spots to the nearest supplied cell centroid.


In [ ]:
def open_stack(path: Path):
    """Open a TIFF stack as a memory-mapped array when possible."""
    arr = tifffile.memmap(path)
    if arr.ndim == 2:
        arr = arr[np.newaxis, ...]
    if arr.ndim != 3:
        raise ValueError(f'Expected image stack as (C,Y,X), got {arr.shape} from {path}')
    return arr

def apply_roi(img2d: np.ndarray, roi_yx):
    if roi_yx is None:
        return img2d
    ys, xs = roi_yx
    return img2d[ys, xs]

def roi_offset(roi_yx):
    if roi_yx is None:
        return 0, 0
    ys, xs = roi_yx
    return (0 if ys.start is None else ys.start, 0 if xs.start is None else xs.start)

def image_qa(img: np.ndarray, label: str) -> dict:
    vals = np.asarray(img)
    finite = vals[np.isfinite(vals)]
    med = float(np.median(finite))
    mad = float(np.median(np.abs(finite - med)))
    p = np.percentile(finite, [1, 99, 99.9, 99.99])
    sat = float((finite >= P.hotpx_sat_value).mean())
    dr = float(p[3] / max(med, 1.0))
    snr = float((p[2] - med) / max(1.4826 * mad, 1.0))
    fg = float((finite > 5 * max(med, 1.0)).mean())
    return dict(label=label, shape=str(vals.shape), dtype=str(vals.dtype), min=float(finite.min()),
                median=med, mean=float(finite.mean()), max=float(finite.max()), p1=float(p[0]),
                p99=float(p[1]), p999=float(p[2]), p9999=float(p[3]), sat_frac=sat,
                dynamic_range=dr, foreground_frac=fg, snr_proxy=snr)

def log_filter(img: np.ndarray) -> np.ndarray:
    # Bright puncta become positive after negating gaussian_laplace.
    return -ndi.gaussian_laplace(img.astype(np.float32, copy=False), sigma=P.log_sigma)

def greedy_nms(coords_ij: np.ndarray, scores: np.ndarray, min_distance: float) -> np.ndarray:
    if len(coords_ij) == 0:
        return coords_ij
    order = np.argsort(scores)[::-1]
    coords = coords_ij[order]
    keep = []
    min_d2 = float(min_distance) ** 2
    for pt in coords:
        if not keep:
            keep.append(pt)
            continue
        prev = np.asarray(keep)
        d2 = np.sum((prev - pt) ** 2, axis=1)
        if np.all(d2 >= min_d2):
            keep.append(pt)
    return np.asarray(keep, dtype=np.int64)

def find_spots_from_score(score: np.ndarray, threshold: float) -> pd.DataFrame:
    local_max = score == ndi.maximum_filter(score, size=P.peak_width, mode='nearest')
    above = score >= threshold
    coords = np.argwhere(local_max & above)
    raw_scores = score[coords[:, 0], coords[:, 1]] if len(coords) else np.array([], dtype=np.float32)
    kept = greedy_nms(coords, raw_scores, P.nms_min_distance)
    kept_scores = score[kept[:, 0], kept[:, 1]] if len(kept) else np.array([], dtype=np.float32)
    return pd.DataFrame({'i': kept[:, 0] if len(kept) else [],
                         'j': kept[:, 1] if len(kept) else [],
                         'score': kept_scores})

def load_cell_features(path: Path, roi_yx=None) -> pd.DataFrame:
    usecols = None
    df = pd.read_csv(path, usecols=usecols)
    required = {'label', 'y', 'x'}
    if not required.issubset(df.columns):
        raise ValueError(f'{path} must contain columns {required}; found {df.columns[:20].tolist()}')
    if roi_yx is not None:
        ys, xs = roi_yx
        y0 = 0 if ys.start is None else ys.start
        y1 = np.inf if ys.stop is None else ys.stop
        x0 = 0 if xs.start is None else xs.start
        x1 = np.inf if xs.stop is None else xs.stop
        df = df[(df['y'] >= y0) & (df['y'] < y1) & (df['x'] >= x0) & (df['x'] < x1)].copy()
        df['y'] -= y0
        df['x'] -= x0
    return df

def assign_spots_to_nearest_cell(spots: pd.DataFrame, cells: pd.DataFrame) -> pd.DataFrame:
    out = spots.copy()
    out['cell'] = 0
    out['cell_distance_px'] = np.nan
    if len(out) == 0 or len(cells) == 0:
        return out
    tree = cKDTree(cells[['y', 'x']].to_numpy(float))
    dist, idx = tree.query(out[['i', 'j']].to_numpy(float), distance_upper_bound=P.cell_assignment_radius_px)
    ok = np.isfinite(dist) & (idx < len(cells))
    labels = cells['label'].to_numpy()
    out.loc[ok, 'cell'] = labels[idx[ok]]
    out.loc[ok, 'cell_distance_px'] = dist[ok]
    return out

def extract_marker_values(stack, markers: List[str], marker_to_idx: Dict[str, int], spots: pd.DataFrame) -> pd.DataFrame:
    vals = pd.DataFrame(index=spots.index)
    if len(spots) == 0:
        return vals
    ii = spots['i'].to_numpy(int)
    jj = spots['j'].to_numpy(int)
    for marker in markers:
        img = apply_roi(stack[marker_to_idx[marker]], P.roi_yx)
        raw_max = ndi.maximum_filter(img.astype(np.float32, copy=False), size=P.max_filter_width, mode='nearest')
        log_img = log_filter(img)
        log_max = ndi.maximum_filter(log_img, size=P.max_filter_width, mode='nearest')
        vals[f'raw_{marker}'] = raw_max[ii, jj]
        vals[f'log_{marker}'] = log_max[ii, jj]
    return vals


# Marker-specific detection helpers used by the calibrated Bit_1 workflow.
def marker_specific_detection_enabled() -> bool:
    return bool(globals().get('PER_MARKER_DETECTION_PARAMS'))


def require_marker_specific_detection_params(context: str):
    if not marker_specific_detection_enabled():
        raise RuntimeError(
            f'{context} requires marker-specific detection thresholds. '
            'Run section 3B through marker_calibration_results before continuing.'
        )


def marker_detection_threshold(marker: str) -> float:
    params = globals().get('PER_MARKER_DETECTION_PARAMS', {})
    if marker in params:
        return float(params[marker]['threshold_peaks'])
    return float(P.threshold_peaks)


def find_marker_specific_spots(stack, markers: List[str], marker_to_idx: Dict[str, int], roi,
                               hot_mask: Optional[np.ndarray] = None,
                               tissue_mask: Optional[np.ndarray] = None) -> pd.DataFrame:
    rows = []
    for marker in markers:
        img = stack[marker_to_idx[marker], roi[0], roi[1]]
        score = log_filter(img)
        if hot_mask is not None:
            score[hot_mask] = 0
        if tissue_mask is not None:
            score[~tissue_mask] = 0
        threshold = marker_detection_threshold(marker)
        spots = find_spots_from_score(score, threshold)
        if len(spots):
            spots['detected_marker'] = marker
            spots['marker_threshold_peaks'] = threshold
            spots['detection_score_norm'] = spots['score'] / max(threshold, 1e-6)
            rows.append(spots)

    if not rows:
        return pd.DataFrame(columns=['i', 'j', 'score', 'detected_marker',
                                     'marker_threshold_peaks', 'detection_score_norm'])

    all_spots = pd.concat(rows, ignore_index=True)
    coords = all_spots[['i', 'j']].to_numpy(int)
    norm_scores = all_spots['detection_score_norm'].to_numpy(float)
    kept_coords = greedy_nms(coords, norm_scores, P.nms_min_distance)
    if len(kept_coords) == 0:
        return all_spots.iloc[0:0].copy()

    # Convert kept coordinates back to row indices. Coordinates are unique enough
    # after per-marker NMS; when duplicated, keep the highest normalized score.
    kept = []
    used = np.zeros(len(all_spots), dtype=bool)
    for pt in kept_coords:
        same = np.where((coords[:, 0] == pt[0]) & (coords[:, 1] == pt[1]) & ~used)[0]
        if len(same):
            best = same[np.argmax(norm_scores[same])]
            kept.append(best)
            used[best] = True
    return all_spots.iloc[kept].reset_index(drop=True)


def active_detection_params() -> dict:
    return dict(
        source=globals().get('DETECTION_PARAMS_SOURCE', 'manual/default P values'),
        log_sigma=P.log_sigma,
        peak_width=P.peak_width,
        threshold_peaks=P.threshold_peaks,
        nms_min_distance=P.nms_min_distance,
        max_filter_width=P.max_filter_width,
        hotpx_count_threshold=P.hotpx_count_threshold,
        hotpx_sat_value=P.hotpx_sat_value,
        marker_specific_thresholds=marker_specific_detection_enabled() if 'marker_specific_detection_enabled' in globals() else False,
    )


def log_active_detection_params(context: str):
    params = active_detection_params()
    log.info(
        f'{context}: using detector params from {params["source"]}: '
        f'log_sigma={params["log_sigma"]:g}, peak_width={params["peak_width"]}, '
        f'threshold_peaks={params["threshold_peaks"]:g}, nms_min_distance={params["nms_min_distance"]}, '
        f'marker_specific_thresholds={params["marker_specific_thresholds"]}'
    )
    return params


## 4. Calibrate Bit_1 detection from PBS and treated spleen controls

The original calibration code is retained. It uses sampled fields from `reg000`
as the negative control and `reg001` as the positive control, then writes the
selected tabular parameter summaries. It does not save image files.


In [ ]:
# Calibrate detector parameters using reg000 as PBS negative control and reg001 as positive control.
# Run this after the visual sanity check and before 4A / full-region detection.
CALIB_NEGATIVE_REGION = 'reg000'
CALIB_POSITIVE_REGION = 'reg001'
CALIB_N_FIELDS_PER_GROUP = 4
CALIB_FIELD_SIZE = 1024
CALIB_RANDOM_SEED = 23
CALIB_MIN_TISSUE_FRACTION = 0.10

# Small grid for interactive calibration. Add values here if the visual sanity plots suggest it.
CALIB_LOG_SIGMAS = [0.8, 1.0, 1.2]
CALIB_PEAK_WIDTHS = [11, 15, 21, 31]
CALIB_NMS_DISTANCE_FRACTION = 0.45

# Selection policy. PBS should be quiet; positive should still have enough local maxima above threshold.
CALIB_MAX_NEG_PEAKS_PER_MPX = 5.0
CALIB_MIN_POS_PEAKS_PER_MPX = 1.0
CALIB_MIN_POS_PEAKS_TOTAL = 10


def calib_info_by_region(region: str) -> dict:
    matches = [info for info in region_info if info['region'] == region]
    if not matches:
        raise ValueError(f'Could not find {region!r} in region_info')
    return matches[0]


def calib_sample_fields(info: dict, n_fields: int, field_size: int, rng) -> List[Tuple[int, int]]:
    H, W = int(info['shape'][1]), int(info['shape'][2])
    field_size = min(int(field_size), H, W)
    tissue_mask = np.load(info['tissue_mask'], mmap_mode='r') if info['tissue_mask'].exists() else None
    fields = []
    max_attempts = max(200, n_fields * 150)
    for _ in range(max_attempts):
        if len(fields) >= n_fields:
            break
        y0 = int(rng.integers(0, H - field_size + 1))
        x0 = int(rng.integers(0, W - field_size + 1))
        if tissue_mask is not None:
            crop_mask = tissue_mask[y0:y0 + field_size, x0:x0 + field_size]
            if float(np.mean(crop_mask)) < CALIB_MIN_TISSUE_FRACTION:
                continue
        fields.append((y0, x0))
    if len(fields) < n_fields:
        log.warning(f"{info['region']}: sampled only {len(fields)}/{n_fields} calibration fields")
    return fields


def calib_score_image(stack, markers: List[str], marker_to_idx: Dict[str, int],
                      y0: int, x0: int, field_size: int, log_sigma: float) -> np.ndarray:
    roi = (slice(y0, y0 + field_size), slice(x0, x0 + field_size))
    first_img = stack[marker_to_idx[markers[0]], roi[0], roi[1]]
    hot_count = np.zeros(first_img.shape, dtype=np.uint8)
    for marker in markers:
        img = stack[marker_to_idx[marker], roi[0], roi[1]]
        hot_count += (img >= P.hotpx_sat_value)
    hot_mask = hot_count >= P.hotpx_count_threshold

    mean = np.zeros(first_img.shape, dtype=np.float32)
    m2 = np.zeros(first_img.shape, dtype=np.float32)
    n = 0
    for marker in markers:
        img = stack[marker_to_idx[marker], roi[0], roi[1]]
        log_img = -ndi.gaussian_laplace(img.astype(np.float32, copy=False), sigma=log_sigma)
        log_img[hot_mask] = 0
        n += 1
        delta = log_img - mean
        mean += delta / n
        m2 += delta * (log_img - mean)

    if len(markers) == 1:
        score = log_img.astype(np.float32, copy=False)
    else:
        score = np.sqrt(m2 / max(n - 1, 1)).astype(np.float32, copy=False)
    score[hot_mask] = 0
    return score


def calib_peak_scores(score: np.ndarray, peak_width: int) -> np.ndarray:
    local_max = score == ndi.maximum_filter(score, size=peak_width, mode='nearest')
    peaks = score[local_max & (score > 0)]
    return peaks[np.isfinite(peaks)]


def calib_candidate_thresholds(neg_peaks: np.ndarray, pos_peaks: np.ndarray) -> np.ndarray:
    vals = []
    if len(neg_peaks):
        vals.extend(np.percentile(neg_peaks, [90, 95, 97, 98, 99, 99.5, 99.9, 99.99]).tolist())
    if len(pos_peaks):
        vals.extend(np.percentile(pos_peaks, [10, 25, 50, 75, 90, 95, 99]).tolist())
    vals.extend([25, 50, 75, 100, 150, 200, 300, 500, 800, float(P.threshold_peaks)])
    vals = np.asarray([v for v in vals if np.isfinite(v) and v > 0], dtype=float)
    if vals.size == 0:
        return np.asarray([float(P.threshold_peaks)], dtype=float)
    return np.unique(np.round(vals, 3))


def calibrate_detection_parameters():
    neg_info = calib_info_by_region(CALIB_NEGATIVE_REGION)
    pos_info = calib_info_by_region(CALIB_POSITIVE_REGION)
    rng = np.random.default_rng(CALIB_RANDOM_SEED)
    field_size = min(CALIB_FIELD_SIZE, int(neg_info['shape'][1]), int(neg_info['shape'][2]),
                     int(pos_info['shape'][1]), int(pos_info['shape'][2]))

    neg_stack = open_stack(neg_info['image'])
    pos_stack = open_stack(pos_info['image'])
    neg_marker_to_idx = {m: i for i, m in enumerate(neg_info['markers'])}
    pos_marker_to_idx = {m: i for i, m in enumerate(pos_info['markers'])}
    markers = [m for m in BARCODE_MARKERS if m in neg_marker_to_idx and m in pos_marker_to_idx]
    if not markers:
        raise ValueError('No shared BARCODE_MARKERS found between negative and positive regions')

    neg_fields = calib_sample_fields(neg_info, CALIB_N_FIELDS_PER_GROUP, field_size, rng)
    pos_fields = calib_sample_fields(pos_info, CALIB_N_FIELDS_PER_GROUP, field_size, rng)
    if not neg_fields or not pos_fields:
        raise ValueError('Calibration needs at least one sampled field from both negative and positive regions')

    log.info(f'Calibration markers: {markers}')
    log.info(f'{CALIB_NEGATIVE_REGION} fields: {neg_fields}')
    log.info(f'{CALIB_POSITIVE_REGION} fields: {pos_fields}')

    rows = []
    neg_area_mpx = len(neg_fields) * field_size * field_size / 1e6
    pos_area_mpx = len(pos_fields) * field_size * field_size / 1e6

    for sigma in CALIB_LOG_SIGMAS:
        neg_scores = [calib_score_image(neg_stack, markers, neg_marker_to_idx, y0, x0, field_size, sigma)
                      for y0, x0 in neg_fields]
        pos_scores = [calib_score_image(pos_stack, markers, pos_marker_to_idx, y0, x0, field_size, sigma)
                      for y0, x0 in pos_fields]

        for peak_width in CALIB_PEAK_WIDTHS:
            neg_peaks = np.concatenate([calib_peak_scores(score, peak_width) for score in neg_scores])
            pos_peaks = np.concatenate([calib_peak_scores(score, peak_width) for score in pos_scores])
            thresholds = calib_candidate_thresholds(neg_peaks, pos_peaks)

            for threshold in thresholds:
                neg_n = int(np.sum(neg_peaks >= threshold)) if len(neg_peaks) else 0
                pos_n = int(np.sum(pos_peaks >= threshold)) if len(pos_peaks) else 0
                neg_rate = neg_n / max(neg_area_mpx, 1e-9)
                pos_rate = pos_n / max(pos_area_mpx, 1e-9)
                enrichment = (pos_rate + 0.1) / (neg_rate + 0.1)
                passes_policy = (
                    neg_rate <= CALIB_MAX_NEG_PEAKS_PER_MPX
                    and pos_rate >= CALIB_MIN_POS_PEAKS_PER_MPX
                    and pos_n >= CALIB_MIN_POS_PEAKS_TOTAL
                )
                # Favor positive retention, strong positive/PBS enrichment, and quiet PBS controls.
                selection_score = (np.log1p(pos_rate) * np.log1p(enrichment)) - np.log1p(neg_rate)
                if passes_policy:
                    selection_score += 1000.0
                rows.append(dict(log_sigma=sigma, peak_width=peak_width, threshold_peaks=float(threshold),
                                 neg_peaks_ge_threshold=neg_n, pos_peaks_ge_threshold=pos_n,
                                 neg_peaks_per_mpx=neg_rate, pos_peaks_per_mpx=pos_rate,
                                 enrichment=enrichment, passes_policy=passes_policy,
                                 selection_score=selection_score,
                                 neg_peak_p99=float(np.percentile(neg_peaks, 99)) if len(neg_peaks) else 0.0,
                                 pos_peak_p50=float(np.percentile(pos_peaks, 50)) if len(pos_peaks) else 0.0,
                                 pos_peak_p99=float(np.percentile(pos_peaks, 99)) if len(pos_peaks) else 0.0))

    calib_df = pd.DataFrame(rows).sort_values(
        ['selection_score', 'pos_peaks_per_mpx', 'enrichment'], ascending=False).reset_index(drop=True)
    if calib_df.empty:
        raise ValueError('Calibration produced no candidate parameter rows')

    best = calib_df.iloc[0]
    # These detector parameters are global for the barcode panel: the same
    # LoG sigma, peak width, threshold, and NMS distance are used for every
    # barcode/RCA channel. The channel-specific part comes later, when each
    # accepted spot is assigned to the marker with the strongest LoG signal.
    P.log_sigma = float(best['log_sigma'])
    P.peak_width = int(best['peak_width'])
    P.threshold_peaks = float(best['threshold_peaks'])
    P.nms_min_distance = max(3, int(round(P.peak_width * CALIB_NMS_DISTANCE_FRACTION)))

    global SELECTED_DETECTION_PARAMS, DETECTION_PARAMS_SOURCE
    DETECTION_PARAMS_SOURCE = f'calibrated from {CALIB_NEGATIVE_REGION} PBS vs {CALIB_POSITIVE_REGION} positive'
    SELECTED_DETECTION_PARAMS = dict(
        source=DETECTION_PARAMS_SOURCE,
        log_sigma=P.log_sigma,
        peak_width=P.peak_width,
        threshold_peaks=P.threshold_peaks,
        nms_min_distance=P.nms_min_distance,
        max_filter_width=P.max_filter_width,
        hotpx_count_threshold=P.hotpx_count_threshold,
        hotpx_sat_value=P.hotpx_sat_value,
        negative_region=CALIB_NEGATIVE_REGION,
        positive_region=CALIB_POSITIVE_REGION,
        negative_peaks_per_mpx=float(best['neg_peaks_per_mpx']),
        positive_peaks_per_mpx=float(best['pos_peaks_per_mpx']),
        enrichment=float(best['enrichment']),
    )

    out_path = OUTDIR / 'calibrated_detection_parameters.csv'
    json_path = OUTDIR / 'selected_detection_parameters.json'
    calib_df.to_csv(out_path, index=False)
    json_path.write_text(json.dumps(SELECTED_DETECTION_PARAMS, indent=2))
    log.info(f'Wrote calibration table to {out_path}')
    log.info(f'Wrote selected parameter JSON to {json_path}')
    log.info(
        'Selected calibrated params: '
        f'log_sigma={P.log_sigma:g}, peak_width={P.peak_width}, '
        f'threshold_peaks={P.threshold_peaks:g}, nms_min_distance={P.nms_min_distance}; '
        f'PBS={best["neg_peaks_per_mpx"]:.2f} peaks/Mpx, '
        f'positive={best["pos_peaks_per_mpx"]:.2f} peaks/Mpx, '
        f'enrichment={best["enrichment"]:.1f}x'
    )
    return calib_df


calibration_results = calibrate_detection_parameters()
calibration_results.head(20)

# Marker-specific threshold calibration.
# Shared parameters: log_sigma, peak_width, nms_min_distance.
# Per-marker parameter: threshold_peaks. This helps dim barcode markers without
# letting bright markers set the threshold for the whole panel.
CALIB_MARKER_MAX_NEG_PEAKS_PER_MPX = 5.0
CALIB_MARKER_MIN_POS_PEAKS_PER_MPX = 0.5
CALIB_MARKER_MIN_POS_PEAKS_TOTAL = 3
CALIB_NEGATIVE_MARKER_PERCENTILE = 99.9


def calib_marker_peak_scores(stack, markers: List[str], marker_to_idx: Dict[str, int],
                             fields: List[Tuple[int, int]], field_size: int,
                             marker: str) -> np.ndarray:
    out = []
    for y0, x0 in fields:
        roi = (slice(y0, y0 + field_size), slice(x0, x0 + field_size))
        first_img = stack[marker_to_idx[markers[0]], roi[0], roi[1]]
        hot_count = np.zeros(first_img.shape, dtype=np.uint8)
        for m in markers:
            img = stack[marker_to_idx[m], roi[0], roi[1]]
            hot_count += (img >= P.hotpx_sat_value)
        hot_mask = hot_count >= P.hotpx_count_threshold

        img = stack[marker_to_idx[marker], roi[0], roi[1]]
        score = log_filter(img)
        score[hot_mask] = 0
        peaks = calib_peak_scores(score, P.peak_width)
        if len(peaks):
            out.append(peaks)
    return np.concatenate(out) if out else np.array([], dtype=np.float32)


def choose_marker_threshold(marker: str, neg_peaks: np.ndarray, pos_peaks: np.ndarray,
                            marker_is_expected_positive: bool, neg_area_mpx: float,
                            pos_area_mpx: float) -> dict:
    if len(neg_peaks):
        neg_conservative = float(np.percentile(neg_peaks, CALIB_NEGATIVE_MARKER_PERCENTILE))
    else:
        neg_conservative = float(P.threshold_peaks)

    if not marker_is_expected_positive:
        # For markers expected to be OFF in the positive control, use a conservative
        # PBS-derived threshold. We do not tune these to positive-control noise.
        thr = max(neg_conservative, float(P.threshold_peaks))
        neg_n = int(np.sum(neg_peaks >= thr)) if len(neg_peaks) else 0
        pos_n = int(np.sum(pos_peaks >= thr)) if len(pos_peaks) else 0
        return dict(marker=marker, expected_positive_in_control=False,
                    threshold_peaks=thr, threshold_source='pbs_conservative_off_bit',
                    neg_peaks_ge_threshold=neg_n, pos_peaks_ge_threshold=pos_n,
                    neg_peaks_per_mpx=neg_n / max(neg_area_mpx, 1e-9),
                    pos_peaks_per_mpx=pos_n / max(pos_area_mpx, 1e-9),
                    enrichment=np.nan,
                    neg_peak_p99=float(np.percentile(neg_peaks, 99)) if len(neg_peaks) else 0.0,
                    neg_peak_p999=float(np.percentile(neg_peaks, 99.9)) if len(neg_peaks) else 0.0,
                    pos_peak_p50=float(np.percentile(pos_peaks, 50)) if len(pos_peaks) else 0.0,
                    pos_peak_p99=float(np.percentile(pos_peaks, 99)) if len(pos_peaks) else 0.0)

    thresholds = calib_candidate_thresholds(neg_peaks, pos_peaks)
    rows = []
    for thr in thresholds:
        neg_n = int(np.sum(neg_peaks >= thr)) if len(neg_peaks) else 0
        pos_n = int(np.sum(pos_peaks >= thr)) if len(pos_peaks) else 0
        neg_rate = neg_n / max(neg_area_mpx, 1e-9)
        pos_rate = pos_n / max(pos_area_mpx, 1e-9)
        enrichment = (pos_rate + 0.1) / (neg_rate + 0.1)
        passes_policy = (
            neg_rate <= CALIB_MARKER_MAX_NEG_PEAKS_PER_MPX
            and pos_rate >= CALIB_MARKER_MIN_POS_PEAKS_PER_MPX
            and pos_n >= CALIB_MARKER_MIN_POS_PEAKS_TOTAL
        )
        selection_score = (np.log1p(pos_rate) * np.log1p(enrichment)) - np.log1p(neg_rate)
        if passes_policy:
            selection_score += 1000.0
        rows.append(dict(threshold_peaks=float(thr), neg_peaks_ge_threshold=neg_n,
                         pos_peaks_ge_threshold=pos_n, neg_peaks_per_mpx=neg_rate,
                         pos_peaks_per_mpx=pos_rate, enrichment=enrichment,
                         passes_policy=passes_policy, selection_score=selection_score))

    best = pd.DataFrame(rows).sort_values(
        ['selection_score', 'pos_peaks_per_mpx', 'enrichment'], ascending=False).iloc[0].to_dict()
    best.update(dict(marker=marker, expected_positive_in_control=True,
                     threshold_source='pbs_vs_positive_on_bit',
                     neg_peak_p99=float(np.percentile(neg_peaks, 99)) if len(neg_peaks) else 0.0,
                     neg_peak_p999=float(np.percentile(neg_peaks, 99.9)) if len(neg_peaks) else 0.0,
                     pos_peak_p50=float(np.percentile(pos_peaks, 50)) if len(pos_peaks) else 0.0,
                     pos_peak_p99=float(np.percentile(pos_peaks, 99)) if len(pos_peaks) else 0.0))
    return best


def calibrate_marker_specific_thresholds():
    if 'calibration_results' not in globals():
        raise RuntimeError('Run calibrate_detection_parameters() before marker-specific calibration')

    neg_info = calib_info_by_region(CALIB_NEGATIVE_REGION)
    pos_info = calib_info_by_region(CALIB_POSITIVE_REGION)
    rng = np.random.default_rng(CALIB_RANDOM_SEED + 1)
    field_size = min(CALIB_FIELD_SIZE, int(neg_info['shape'][1]), int(neg_info['shape'][2]),
                     int(pos_info['shape'][1]), int(pos_info['shape'][2]))

    neg_stack = open_stack(neg_info['image'])
    pos_stack = open_stack(pos_info['image'])
    neg_marker_to_idx = {m: i for i, m in enumerate(neg_info['markers'])}
    pos_marker_to_idx = {m: i for i, m in enumerate(pos_info['markers'])}
    markers = [m for m in BARCODE_MARKERS if m in neg_marker_to_idx and m in pos_marker_to_idx]
    if not markers:
        raise ValueError('No shared BARCODE_MARKERS found between negative and positive regions')

    expected_positive_markers, expected_negative_markers = marker_sets_for_lnp_name(QC_REFERENCE_LNP)
    expected_positive_markers = [m for m in expected_positive_markers if m in markers]
    expected_negative_markers = [m for m in expected_negative_markers if m in markers]

    neg_fields = calib_sample_fields(neg_info, CALIB_N_FIELDS_PER_GROUP, field_size, rng)
    pos_fields = calib_sample_fields(pos_info, CALIB_N_FIELDS_PER_GROUP, field_size, rng)
    neg_area_mpx = len(neg_fields) * field_size * field_size / 1e6
    pos_area_mpx = len(pos_fields) * field_size * field_size / 1e6

    log.info(f'Marker-specific calibration uses shared shape params: log_sigma={P.log_sigma:g}, peak_width={P.peak_width}')
    log.info(f'Expected ON markers for {QC_REFERENCE_LNP}: {expected_positive_markers}')
    log.info(f'Expected OFF markers for {QC_REFERENCE_LNP}: {expected_negative_markers}')

    rows = []
    for marker in markers:
        neg_peaks = calib_marker_peak_scores(neg_stack, markers, neg_marker_to_idx, neg_fields, field_size, marker)
        pos_peaks = calib_marker_peak_scores(pos_stack, markers, pos_marker_to_idx, pos_fields, field_size, marker)
        rows.append(choose_marker_threshold(
            marker, neg_peaks, pos_peaks, marker in expected_positive_markers,
            neg_area_mpx, pos_area_mpx))

    marker_df = pd.DataFrame(rows)
    marker_df = marker_df.sort_values(['expected_positive_in_control', 'marker'], ascending=[False, True]).reset_index(drop=True)

    global PER_MARKER_DETECTION_PARAMS
    PER_MARKER_DETECTION_PARAMS = {
        row['marker']: dict(threshold_peaks=float(row['threshold_peaks']),
                            expected_positive_in_control=bool(row['expected_positive_in_control']),
                            threshold_source=row['threshold_source'])
        for _, row in marker_df.iterrows()
    }
    SELECTED_DETECTION_PARAMS['marker_specific_thresholds'] = PER_MARKER_DETECTION_PARAMS
    SELECTED_DETECTION_PARAMS['marker_specific_thresholds_enabled'] = True

    out_path = OUTDIR / 'calibrated_marker_detection_parameters.csv'
    json_path = OUTDIR / 'selected_detection_parameters.json'
    marker_df.to_csv(out_path, index=False)
    json_path.write_text(json.dumps(SELECTED_DETECTION_PARAMS, indent=2))
    log.info(f'Wrote marker-specific threshold table to {out_path}')
    log.info('Marker-specific thresholds: ' + ', '.join(
        f'{m}={PER_MARKER_DETECTION_PARAMS[m]["threshold_peaks"]:.3g}' for m in markers))
    return marker_df


marker_calibration_results = calibrate_marker_specific_thresholds()
marker_calibration_results


## 5. Run full-region Bit_1 spot detection

This cell processes the two full registered spleen images, assigns detected Bit_1
spots to cells, and writes per-region spot and cell-count CSV files. It was not
run while preparing this publication copy.


In [ ]:
require_marker_specific_detection_params('full-region detection')
run_params = log_active_detection_params('full-region detection')

all_spots = {}
all_cells = {}
qa_rows = []
spot_qa_rows = []

for r in region_info:
    region = r['region']
    sample = r['sample']
    log.info(f'===== {region} / {sample} =====')
    t_region = tic()

    markers = r['markers']
    marker_to_idx = {m: i for i, m in enumerate(markers)}
    missing_markers = [m for m in BARCODE_MARKERS if m not in marker_to_idx]
    if missing_markers:
        log.warning(f'{region}: missing barcode markers {missing_markers}; skipping them')
    region_markers = [m for m in BARCODE_MARKERS if m in marker_to_idx]

    stack = open_stack(r['image'])
    log.info(f'image shape: {stack.shape}; barcode markers: {region_markers}')

    # QA and hot-pixel count.
    first_img = apply_roi(stack[marker_to_idx[region_markers[0]]], P.roi_yx)
    hot_count = np.zeros(first_img.shape, dtype=np.uint8)
    for marker in region_markers:
        img = apply_roi(stack[marker_to_idx[marker]], P.roi_yx)
        qa_rows.append({'region': region, 'sample': sample, 'marker': marker, **image_qa(img, marker)})
        hot_count += (img >= P.hotpx_sat_value)
    hot_mask = hot_count >= P.hotpx_count_threshold
    n_hot = int(hot_mask.sum())
    log.info(f'hot pixels masked: {n_hot}')

    # Restrict calls to the tissue/polygon mask produced by tissue-extractor, when present.
    # This mask is not a cell-label mask; it is a region-of-interest mask.
    tissue_mask = None
    if r['tissue_mask'].exists():
        tissue_mask = apply_roi(np.load(r['tissue_mask'], mmap_mode='r').astype(bool), P.roi_yx)
        if tissue_mask.shape != first_img.shape:
            log.warning(f"{region}: tissue mask shape {tissue_mask.shape} != image shape {first_img.shape}; not applying tissue mask")
            tissue_mask = None
        else:
            log.info(f"{region}: tissue mask keeps {tissue_mask.mean() * 100:.1f}% of pixels")

    if marker_specific_detection_enabled():
        log.info(f'{region}: using marker-specific detection thresholds')
        full_roi = P.roi_yx if P.roi_yx is not None else (slice(None), slice(None))
        spots = find_marker_specific_spots(stack, region_markers, marker_to_idx, full_roi,
                                           hot_mask=hot_mask, tissue_mask=tissue_mask)
    else:
        # Original shared detector: streaming std across LoG-filtered barcode channels.
        mean = np.zeros(first_img.shape, dtype=np.float32)
        m2 = np.zeros(first_img.shape, dtype=np.float32)
        n = 0
        for marker in region_markers:
            t = tic()
            img = apply_roi(stack[marker_to_idx[marker]], P.roi_yx)
            log_img = log_filter(img)
            log_img[hot_mask] = 0
            if tissue_mask is not None:
                log_img[~tissue_mask] = 0
            n += 1
            delta = log_img - mean
            mean += delta / n
            m2 += delta * (log_img - mean)
            toc(t, f'{region} LoG {marker}')
        if len(region_markers) == 1:
            score = log_img.astype(np.float32, copy=False)
        else:
            score = np.sqrt(m2 / max(n - 1, 1)).astype(np.float32, copy=False)
        score[hot_mask] = 0
        if tissue_mask is not None:
            score[~tissue_mask] = 0
        spots = find_spots_from_score(score, P.threshold_peaks)

    y0, x0 = roi_offset(P.roi_yx)
    spots['i_global'] = spots['i'] + y0
    spots['j_global'] = spots['j'] + x0
    log.info(f'{region}: detected {len(spots)} spots')

    # Extract raw and LoG values at every spot, then call marker by max LoG.
    vals = extract_marker_values(stack, region_markers, marker_to_idx, spots)
    spots = pd.concat([spots, vals], axis=1)
    log_cols = [f'log_{m}' for m in region_markers]
    if len(spots):
        best_idx = np.argmax(spots[log_cols].to_numpy(float), axis=1)
        spots['called_marker'] = [region_markers[i] for i in best_idx]
        spots['called_log_intensity'] = spots[log_cols].to_numpy(float)[np.arange(len(spots)), best_idx]
    else:
        spots['called_marker'] = []
        spots['called_log_intensity'] = []

    # Assign to segmented cells by nearest centroid from Mesmer features.
    cells = load_cell_features(r['features'], roi_yx=P.roi_yx)
    spots = assign_spots_to_nearest_cell(spots, cells)

    # Per-cell spot count matrix.
    assigned = spots[spots['cell'] > 0].copy()
    if len(assigned):
        counts = assigned.pivot_table(index='cell', columns='called_marker', values='score', aggfunc='count', fill_value=0)
        counts = counts.reindex(columns=region_markers, fill_value=0)
        counts['total_spots'] = counts[region_markers].sum(axis=1)
        counts['dominant_marker'] = counts[region_markers].idxmax(axis=1)
        counts['dominant_count'] = counts[region_markers].max(axis=1)
        counts = counts[counts['total_spots'] >= P.min_spots_per_cell].reset_index()
    else:
        counts = pd.DataFrame(columns=['cell', *region_markers, 'total_spots', 'dominant_marker', 'dominant_count'])

    spot_path = OUTDIR / f'spots_{region}.csv'
    cell_path = OUTDIR / f'cell_spot_counts_{region}.csv'
    spots.to_csv(spot_path, index=False)
    counts.to_csv(cell_path, index=False)
    log.info(f'wrote {spot_path.name} and {cell_path.name}')

    all_spots[region] = spots
    all_cells[region] = counts
    spot_qa_rows.append(dict(region=region, sample=sample, n_markers=len(region_markers),
                             detection_source=run_params['source'],
                             log_sigma=run_params['log_sigma'], peak_width=run_params['peak_width'],
                             threshold_peaks=run_params['threshold_peaks'],
                             nms_min_distance=run_params['nms_min_distance'],
                             marker_specific_thresholds=run_params.get('marker_specific_thresholds', False),
                             n_hot_pixels=n_hot, n_spots=len(spots),
                             n_spots_assigned=int((spots['cell'] > 0).sum()) if len(spots) else 0,
                             n_callable_cells=len(counts)))
    toc(t_region, f'{region} total')

qa_per_image = pd.DataFrame(qa_rows)
qa_spot_detection = pd.DataFrame(spot_qa_rows)
qa_per_image.to_csv(OUTDIR / 'qa_per_image.csv', index=False)
qa_spot_detection.to_csv(OUTDIR / 'qa_spot_detection.csv', index=False)
qa_spot_detection


## 6. Combine the two spleen regions

Only combined CSV summaries are written. The exploratory quick plot and the
PNG-producing visual-QC sections were removed.


In [ ]:
combined_spots = []
combined_cells = []
for region, df in all_spots.items():
    tmp = df.copy()
    tmp.insert(0, 'region', region)
    combined_spots.append(tmp)
for region, df in all_cells.items():
    tmp = df.copy()
    tmp.insert(0, 'region', region)
    combined_cells.append(tmp)

combined_spots = pd.concat(combined_spots, ignore_index=True) if combined_spots else pd.DataFrame()
combined_cells = pd.concat(combined_cells, ignore_index=True) if combined_cells else pd.DataFrame()
combined_spots.to_csv(OUTDIR / 'spots_all_regions.csv', index=False)
combined_cells.to_csv(OUTDIR / 'cell_spot_counts_all_regions.csv', index=False)

summary = (combined_spots
           .pivot_table(index='region', columns='called_marker', values='score', aggfunc='count', fill_value=0)
           .reindex(columns=BARCODE_MARKERS, fill_value=0))
summary.to_csv(OUTDIR / 'spot_counts_by_marker.csv')
summary


## 7. Build the final cell-level Bit_1 call table

A cell is Bit_1-positive when it has at least the configured number of assigned
Bit_1 spots. The published run used the calibrated single-channel detector and
the threshold recorded in the generated parameter tables.


In [ ]:
require_marker_specific_detection_params('final cell-level analysis')
if 'qa_spot_detection' in globals() and not qa_spot_detection.empty:
    if 'marker_specific_thresholds' not in qa_spot_detection.columns or not qa_spot_detection['marker_specific_thresholds'].fillna(False).all():
        raise RuntimeError('Final analysis requires spots generated with marker-specific thresholds. Rerun section 4 first.')

def hamming_distance(a: str, b: str) -> int:
    if len(a) != len(b):
        return max(len(a), len(b))
    return sum(x != y for x, y in zip(a, b))

def tolerant_lnp_match(barcode: str, total_spots: int) -> dict:
    if total_spots == 0:
        return dict(lnp_call='no_barcode', barcode_in_library=False,
                    barcode_match_distance=np.nan, barcode_match_status='no_barcode',
                    barcode_excluded=True)
    if not BARCODE_LIBRARY:
        return dict(lnp_call='unmapped', barcode_in_library=False,
                    barcode_match_distance=np.nan, barcode_match_status='unmapped',
                    barcode_excluded=False)

    distances = [(lib_bc, name, hamming_distance(barcode, lib_bc))
                 for lib_bc, name in BARCODE_LIBRARY.items()]
    min_dist = min(d for _, _, d in distances)
    candidates = [(lib_bc, name, d) for lib_bc, name, d in distances
                  if d == min_dist and d <= P.barcode_max_hamming_distance]

    if len(candidates) == 1:
        lib_bc, name, dist = candidates[0]
        return dict(lnp_call=name, barcode_in_library=(dist == 0),
                    barcode_match_distance=dist,
                    barcode_match_status='exact' if dist == 0 else 'tolerant',
                    barcode_excluded=False)
    if len(candidates) > 1:
        return dict(lnp_call='ambiguous_mixed', barcode_in_library=False,
                    barcode_match_distance=min_dist, barcode_match_status='ambiguous_mixed',
                    barcode_excluded=True)
    return dict(lnp_call='unmapped', barcode_in_library=False,
                barcode_match_distance=min_dist, barcode_match_status='unmapped',
                barcode_excluded=False)

def build_cell_analysis_table_for_region(region: str, info: dict, spots: pd.DataFrame) -> pd.DataFrame:
    cells = pd.read_csv(info['features'])
    if 'label' not in cells.columns or 'x' not in cells.columns or 'y' not in cells.columns:
        raise ValueError(f"{info['features']} must contain label, x, and y columns")

    cells = cells.copy()
    cells.insert(0, 'region', region)
    cells = cells.rename(columns={'label': 'cell'})

    # Count called barcode-marker spots assigned to each cell.
    assigned = spots[spots['cell'] > 0].copy() if len(spots) else pd.DataFrame()
    if len(assigned):
        counts = assigned.pivot_table(
            index='cell',
            columns='called_marker',
            values='score',
            aggfunc='count',
            fill_value=0,
        )
        counts = counts.reindex(columns=BARCODE_MARKERS, fill_value=0)
        counts.columns = [f'spot_{m}' for m in counts.columns]
        counts = counts.reset_index()
    else:
        counts = pd.DataFrame({'cell': cells['cell']})
        for marker in BARCODE_MARKERS:
            counts[f'spot_{marker}'] = 0

    out = cells.merge(counts, on='cell', how='left')
    spot_cols = [f'spot_{m}' for m in BARCODE_MARKERS]
    for col in spot_cols:
        if col not in out.columns:
            out[col] = 0
        out[col] = out[col].fillna(0).astype(int)

    # Binary barcode bits from spot counts.
    bit_cols = []
    for marker in BARCODE_MARKERS:
        bit_col = f'bit_{marker}'
        spot_col = f'spot_{marker}'
        out[bit_col] = (out[spot_col] >= P.min_spots_for_bit).astype(int)
        bit_cols.append(bit_col)

    out['barcode'] = out[bit_cols].astype(str).agg(''.join, axis=1)
    out['total_barcode_spots'] = out[spot_cols].sum(axis=1)
    out['n_positive_bits'] = out[bit_cols].sum(axis=1)

    # Dominant marker and simple confidence summaries.
    spot_matrix = out[spot_cols].to_numpy(int)
    if len(out):
        dominant_idx = np.argmax(spot_matrix, axis=1)
        dominant_counts = spot_matrix[np.arange(len(out)), dominant_idx]
        out['dominant_barcode_marker'] = [BARCODE_MARKERS[i] if total > 0 else 'none'
                                          for i, total in zip(dominant_idx, out['total_barcode_spots'])]
        out['dominant_barcode_count'] = dominant_counts
        out['dominant_barcode_fraction'] = np.where(
            out['total_barcode_spots'] > 0,
            out['dominant_barcode_count'] / out['total_barcode_spots'],
            0.0,
        )
    else:
        out['dominant_barcode_marker'] = []
        out['dominant_barcode_count'] = []
        out['dominant_barcode_fraction'] = []

    positive_spot_sum = np.zeros(len(out), dtype=int)
    for marker in BARCODE_MARKERS:
        positive_spot_sum += out[f'spot_{marker}'].where(out[f'bit_{marker}'] == 1, 0).to_numpy(int)
    out['barcode_confidence'] = np.where(
        out['total_barcode_spots'] > 0,
        positive_spot_sum / out['total_barcode_spots'],
        0.0,
    )

    match_rows = [tolerant_lnp_match(bc, int(total))
                  for bc, total in zip(out['barcode'], out['total_barcode_spots'])]
    match_df = pd.DataFrame(match_rows, index=out.index)
    for col in match_df.columns:
        out[col] = match_df[col]
    out['lnp_positive'] = (
        (~out['barcode_excluded'])
        & ~out['lnp_call'].isin(['no_barcode', 'unmapped', 'ambiguous_mixed'])
    )

    # Put downstream-critical columns near the front while keeping all original marker features.
    front = [
        'region', 'cell', 'x', 'y', 'area', 'eccentricity',
        'barcode', 'lnp_call', 'lnp_positive', 'barcode_in_library',
        'barcode_match_distance', 'barcode_match_status', 'barcode_excluded',
        'total_barcode_spots', 'n_positive_bits',
        'dominant_barcode_marker', 'dominant_barcode_count', 'dominant_barcode_fraction',
        'barcode_confidence',
    ]
    front = [c for c in front if c in out.columns]
    rest = [c for c in out.columns if c not in front]
    return out[front + rest]

cell_analysis_tables = []
for info in region_info:
    region = info['region']
    table = build_cell_analysis_table_for_region(region, info, all_spots.get(region, pd.DataFrame()))
    table.to_csv(OUTDIR / f'cell_analysis_table_{region}.csv', index=False)
    cell_analysis_tables.append(table)
    log.info(f"{region}: wrote cell_analysis_table_{region}.csv with {len(table)} cells")

cell_analysis_table = pd.concat(cell_analysis_tables, ignore_index=True)
cell_analysis_table.to_csv(OUTDIR / 'cell_analysis_table_all_regions.csv', index=False)
log.info(f"Wrote {OUTDIR / 'cell_analysis_table_all_regions.csv'} with {len(cell_analysis_table)} cells")

# Alternate one-LNP analysis for this round: call every A20+ cell as LNP_A+.
# This ignores the full 12-bit barcode and is useful while only one LNP is present.
A20_ONLY_MARKER = 'A20'
A20_ONLY_MIN_SPOTS = P.min_spots_for_bit
A20_ONLY_LNP_NAME = 'LNP_A'
A20_ONLY_SPOT_COL = f'spot_{A20_ONLY_MARKER}'
if A20_ONLY_SPOT_COL not in cell_analysis_table.columns:
    raise ValueError(f'{A20_ONLY_SPOT_COL} is missing from cell_analysis_table; check BARCODE_MARKERS')

cell_analysis_table_a20_only = cell_analysis_table.copy()
cell_analysis_table_a20_only['a20_only_marker'] = A20_ONLY_MARKER
cell_analysis_table_a20_only['a20_only_min_spots'] = A20_ONLY_MIN_SPOTS
cell_analysis_table_a20_only['a20_positive'] = cell_analysis_table_a20_only[A20_ONLY_SPOT_COL] >= A20_ONLY_MIN_SPOTS
cell_analysis_table_a20_only['lnp_call'] = np.where(
    cell_analysis_table_a20_only['a20_positive'], A20_ONLY_LNP_NAME, 'no_A20')
cell_analysis_table_a20_only['lnp_positive'] = cell_analysis_table_a20_only['a20_positive']
cell_analysis_table_a20_only['barcode_match_status'] = np.where(
    cell_analysis_table_a20_only['a20_positive'], 'A20_only_positive', 'A20_only_negative')
cell_analysis_table_a20_only['barcode_excluded'] = False
cell_analysis_table_a20_only['barcode_in_library'] = cell_analysis_table_a20_only['a20_positive']
cell_analysis_table_a20_only['barcode_match_distance'] = np.nan

front_a20 = [
    'region', 'cell', 'x', 'y', 'area', 'eccentricity',
    'lnp_call', 'lnp_positive', 'a20_positive', A20_ONLY_SPOT_COL,
    'a20_only_min_spots', 'a20_only_marker',
    'barcode', 'total_barcode_spots', 'n_positive_bits',
    'dominant_barcode_marker', 'dominant_barcode_count', 'dominant_barcode_fraction',
]
front_a20 = [c for c in front_a20 if c in cell_analysis_table_a20_only.columns]
rest_a20 = [c for c in cell_analysis_table_a20_only.columns if c not in front_a20]
cell_analysis_table_a20_only = cell_analysis_table_a20_only[front_a20 + rest_a20]
cell_analysis_table_a20_only.to_csv(OUTDIR / 'cell_analysis_table_Bit_1_spleen_regions.csv', index=False)
log.info(f"Wrote {OUTDIR / 'cell_analysis_table_Bit_1_spleen_regions.csv'} with {len(cell_analysis_table_a20_only)} cells")

cell_analysis_summary_a20_only = (cell_analysis_table_a20_only
                                  .groupby(['region', 'lnp_call'], dropna=False)
                                  .size()
                                  .reset_index(name='n_cells'))
cell_analysis_summary_a20_only.to_csv(OUTDIR / 'cell_analysis_summary_Bit_1_by_call.csv', index=False)

# Small summary for sanity checking of the full 12-bit barcode call.
cell_analysis_summary = (cell_analysis_table
                         .groupby(['region', 'lnp_call'], dropna=False)
                         .size()
                         .reset_index(name='n_cells'))
cell_analysis_summary.to_csv(OUTDIR / 'cell_analysis_summary_by_lnp_call.csv', index=False)

print('Full 12-bit barcode summary:')
display(cell_analysis_summary)
print('Bit_1-positive summary:')
cell_analysis_summary_a20_only


## CSV outputs

The principal files written to `Generated_Output/Bit_1/` are:

- `spots_reg000.csv` and `spots_reg001.csv`: detected Bit_1 spots;
- `cell_spot_counts_reg000.csv` and `cell_spot_counts_reg001.csv`:
  assigned Bit_1 spot counts per cell;
- `spots_all_regions.csv`: spots combined across the two spleens; and
- `cell_analysis_table_Bit_1_spleen_regions.csv`: final cell-level Bit_1 calls.

The compact precomputed Bit_1 table supplied in `Precomputed_Analysis_Input/`
lets the plotting notebook run without repeating this large-image step.
